# Pure-GNN v3.1 authorized technical preflight

## REQUIRED KAGGLE SETUP

1. Accelerator: **NVIDIA T4 GPU**.
2. Internet: **ON**, required only to clone the public immutable Git tag.
3. Attach Kaggle dataset: **`doduyquynii/fer13-split`**.

This notebook reads only official `train.csv` from the approved dataset mount. It never opens, reads, hashes, discovers, or parses `val.csv` or `test.csv`. It creates no research split and performs no scientific training.


In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/Irthn1311/FER2013_Graph.git'
SOURCE_TAG = 'pure-gnn-v31-preflight-v2'
REQUIRED_GPU_SUBSTRING = 'T4'
REQUIRE_GPU = True
RUN_TESTS = True
RUN_TRAIN_INPUT_VALIDATION = True
RUN_TECHNICAL_PREFLIGHT = True
RUN_RUNTIME_BENCHMARK = True
RUN_RESEARCH_SCREEN = False
TECHNICAL_SEED = 42  # synthetic tests only; not a split/model/training seed
RESEARCH_SPLIT_SEED = None
APPROVED_TRAIN_CANDIDATES = (
    Path('/kaggle/input/fer13-split/fer13-split/train.csv'),
    Path('/kaggle/input/fer13-split/train.csv'),
)
WORKING_ROOT = Path('/kaggle/working')
REPOSITORY_ROOT = WORKING_ROOT / 'FER2013_Graph'
OUTPUT_ROOT = WORKING_ROOT / 'pure_gnn_v31_preflight'
PACKAGE_RELATIVE = Path('research/pure_gnn_v31')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


## Immutable source checkout

In [ ]:
import json, os, re, shutil, subprocess, sys

def run_checked(command, cwd=None, capture=True):
    rendered = [str(value) for value in command]
    print('$', ' '.join(rendered))
    result = subprocess.run(rendered, cwd=cwd, text=True, stdout=subprocess.PIPE if capture else None, stderr=subprocess.STDOUT if capture else None)
    if result.returncode != 0:
        if result.stdout:
            print('\n'.join(result.stdout.splitlines()[-100:]))
        raise RuntimeError(f'Command failed ({result.returncode}): {rendered}')
    return result.stdout or ''

if REPOSITORY_ROOT.exists():
    shutil.rmtree(REPOSITORY_ROOT)
run_checked(['git', 'clone', REPO_URL, str(REPOSITORY_ROOT)])
run_checked(['git', 'fetch', '--tags'], cwd=REPOSITORY_ROOT)
run_checked(['git', 'show-ref', '--verify', f'refs/tags/{SOURCE_TAG}'], cwd=REPOSITORY_ROOT)
tag_commit = run_checked(['git', 'rev-parse', f'refs/tags/{SOURCE_TAG}^{{commit}}'], cwd=REPOSITORY_ROOT).strip()
run_checked(['git', 'checkout', '--detach', SOURCE_TAG], cwd=REPOSITORY_ROOT)
actual_head = run_checked(['git', 'rev-parse', 'HEAD'], cwd=REPOSITORY_ROOT).strip()
if actual_head != tag_commit:
    raise RuntimeError(f'Source lock mismatch: tag={tag_commit}, HEAD={actual_head}')
if run_checked(['git', 'status', '--porcelain'], cwd=REPOSITORY_ROOT).strip():
    raise RuntimeError('Detached source worktree is not clean')
source_lock = {'status': 'PASS', 'source_tag': SOURCE_TAG, 'tag_commit': tag_commit, 'actual_head': actual_head, 'detached_head': True, 'exact_match': True}
(OUTPUT_ROOT / 'source_lock.json').write_text(json.dumps(source_lock, indent=2), encoding='utf-8')
PACKAGE_ROOT = REPOSITORY_ROOT / PACKAGE_RELATIVE


## T4 environment gate

In [ ]:
import platform
import tensorflow as tf

tf.keras.mixed_precision.set_global_policy('float32')
gpu_rows = run_checked(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader']).strip().splitlines()
gpu_names = [row.split(',')[0].strip() for row in gpu_rows if row.strip()]
driver_versions = [row.split(',', 1)[1].strip() for row in gpu_rows if ',' in row]
tf_gpu_devices = [device.name for device in tf.config.list_physical_devices('GPU')]
if REQUIRE_GPU and (not gpu_names or not tf_gpu_devices):
    raise RuntimeError('NVIDIA GPU and TensorFlow GPU visibility are required')
if not any(REQUIRED_GPU_SUBSTRING in name for name in gpu_names):
    raise RuntimeError(f'This snapshot requires T4; detected {gpu_names}')
environment_report = {
    'status': 'PASS', 'gpu_names': gpu_names, 'driver_versions': driver_versions,
    'required_gpu_substring': REQUIRED_GPU_SUBSTRING, 'tensorflow_version': tf.__version__,
    'tf_gpu_devices': tf_gpu_devices, 'cuda_visible_devices': os.environ.get('CUDA_VISIBLE_DEVICES'),
    'dtype_policy': tf.keras.mixed_precision.global_policy().name, 'python': sys.version,
    'platform': platform.platform(), 'distribution_strategy': 'single_device_GPU_0'
}
(OUTPUT_ROOT / 'environment_report.json').write_text(json.dumps(environment_report, indent=2), encoding='utf-8')


## Isolated package installation and import

In [ ]:
import importlib
import site

run_checked([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(PACKAGE_ROOT), '--no-deps'], capture=False)
# A newly written editable-install .pth file is processed only at interpreter
# startup unless the running notebook explicitly re-processes its existing
# site-package directories. This activates installation metadata; it does not
# inject the repository source directory into sys.path.
site_directories = [Path(value).resolve() for value in site.getsitepackages()]
if site.ENABLE_USER_SITE:
    site_directories.append(Path(site.getusersitepackages()).resolve())
for site_directory in dict.fromkeys(site_directories):
    if site_directory.is_dir():
        site.addsitedir(str(site_directory))
importlib.invalidate_caches()
import pure_gnn_v31
package_file = Path(pure_gnn_v31.__file__).resolve()
expected_package_source = (PACKAGE_ROOT / 'src' / 'pure_gnn_v31').resolve()
if expected_package_source not in package_file.parents:
    raise RuntimeError(f'Package imported outside reviewed source: {package_file}')
legacy_tokens = ('lap_gnn', 'ws_hpg', 'tf_cf_hpg', 'tf_ra_hpg')
legacy_loaded = sorted(name for name in sys.modules if any(token in name.lower() for token in legacy_tokens))
if legacy_loaded:
    raise RuntimeError(f'Legacy architecture modules loaded: {legacy_loaded}')
if run_checked(['git', 'status', '--porcelain'], cwd=REPOSITORY_ROOT).strip():
    raise RuntimeError('Package installation modified tracked source')


## Complete bounded package tests

In [ ]:
test_report_path = OUTPUT_ROOT / 'test_report.txt'
if not RUN_TESTS:
    raise PermissionError('This reviewed notebook requires RUN_TESTS=True')
test_result = subprocess.run([sys.executable, '-m', 'pytest', str(PACKAGE_ROOT / 'tests'), '-q'], cwd=PACKAGE_ROOT, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
test_report_path.write_text(test_result.stdout, encoding='utf-8')
print('\n'.join(test_result.stdout.splitlines()[-30:]))
if test_result.returncode != 0:
    raise RuntimeError(f'Bounded tests failed with exit code {test_result.returncode}')


## Official train.csv provenance validation only

In [ ]:
from pure_gnn_v31.data import assert_preflight_train_only_path, validate_official_train_csv

if not RUN_TRAIN_INPUT_VALIDATION:
    raise PermissionError('This reviewed notebook requires train input validation')
for candidate in APPROVED_TRAIN_CANDIDATES:
    assert_preflight_train_only_path(candidate)
existing_train_candidates = [candidate for candidate in APPROVED_TRAIN_CANDIDATES if candidate.is_file()]
if len(existing_train_candidates) != 1:
    raise FileNotFoundError('Require exactly one approved train.csv candidate. Attach Kaggle dataset doduyquynii/fer13-split. Found: ' + repr(existing_train_candidates))
approved_train_csv = existing_train_candidates[0]
train_input_report = validate_official_train_csv(approved_train_csv)
(OUTPUT_ROOT / 'train_input_report.json').write_text(json.dumps(train_input_report, indent=2), encoding='utf-8')
print(json.dumps(train_input_report, indent=2))


## Synthetic technical preflight and T4 runtime benchmark

In [ ]:
from pure_gnn_v31.cli.preflight import run_technical_preflight
from pure_gnn_v31.tools.benchmark_runtime import benchmark_batch_sizes

if not RUN_TECHNICAL_PREFLIGHT or not RUN_RUNTIME_BENCHMARK:
    raise PermissionError('Reviewed technical preflight and runtime benchmark must remain enabled')
preflight_report = run_technical_preflight(str(OUTPUT_ROOT / 'preflight_report.json'), technical_seed=TECHNICAL_SEED)
if preflight_report['status'] != 'PASS':
    raise RuntimeError('Synthetic technical preflight did not pass')
benchmark_report = benchmark_batch_sizes([16, 32, 64], condition='G1', num_warmup=5, num_steps=10, output_path=str(OUTPUT_ROOT / 'benchmark_report.json'), technical_seed=TECHNICAL_SEED)
if benchmark_report['status'] != 'PASS' or benchmark_report['benchmarks'].get('B32', {}).get('status') != 'COMPLETE':
    raise RuntimeError('Mandatory B32 inference plus optimizer-step benchmark is incomplete')


## Scientific execution lock

In [ ]:
if RUN_RESEARCH_SCREEN:
    raise PermissionError('SCIENTIFIC TRAINING IS NOT AUTHORIZED')
print('Scientific screen remains locked; no training or research split was executed.')


## Evidence validation and compact archive

In [ ]:
import hashlib, tarfile

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

source_files = sorted(path for path in PACKAGE_ROOT.rglob('*') if path.is_file() and '.egg-info' not in path.parts and '__pycache__' not in path.parts and '.pytest_cache' not in path.parts)
source_manifest = {'source_tag': SOURCE_TAG, 'commit': actual_head, 'files': {str(path.relative_to(REPOSITORY_ROOT)).replace('\\', '/'): sha256_file(path) for path in source_files}}
(OUTPUT_ROOT / 'source_manifest.json').write_text(json.dumps(source_manifest, indent=2, sort_keys=True), encoding='utf-8')
technical_config = {'technical_seed': TECHNICAL_SEED, 'technical_seed_scope': 'synthetic_only', 'research_split_seed': RESEARCH_SPLIT_SEED, 'mixed_precision_policy': 'float32', 'scientific_training_authorized': False}
(OUTPUT_ROOT / 'technical_config_snapshot.json').write_text(json.dumps(technical_config, indent=2), encoding='utf-8')
parameter_report = preflight_report['parameter_info']
(OUTPUT_ROOT / 'parameter_report.json').write_text(json.dumps(parameter_report, indent=2), encoding='utf-8')
summary = {
    'status': 'PASS', 'source_lock_pass': source_lock['exact_match'], 'environment_pass': environment_report['status'] == 'PASS',
    'tests_pass': test_result.returncode == 0, 'train_input_validation_pass': train_input_report['status'] == 'PASS',
    'technical_preflight_pass': preflight_report['status'] == 'PASS', 'b32_runtime_complete': benchmark_report['benchmarks']['B32']['status'] == 'COMPLETE',
    'opened_data_paths': train_input_report['opened_data_paths'], 'research_split_created': False,
    'scientific_training_performed': False, 'validation_access': False, 'test_access': False,
}
if not all((summary['source_lock_pass'], summary['environment_pass'], summary['tests_pass'], summary['train_input_validation_pass'], summary['technical_preflight_pass'], summary['b32_runtime_complete'])):
    raise RuntimeError(f'Post-run technical evidence invalid: {summary}')
(OUTPUT_ROOT / 'notebook_run_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
required_members = ('source_lock.json', 'environment_report.json', 'train_input_report.json', 'test_report.txt', 'preflight_report.json', 'benchmark_report.json', 'parameter_report.json', 'source_manifest.json', 'technical_config_snapshot.json', 'notebook_run_summary.json')
missing = [name for name in required_members if not (OUTPUT_ROOT / name).is_file()]
if missing:
    raise RuntimeError(f'Missing required evidence: {missing}')
archive_path = WORKING_ROOT / 'pure_gnn_v31_preflight_evidence.tar.gz'
with tarfile.open(archive_path, 'w:gz') as archive:
    for name in required_members:
        archive.add(OUTPUT_ROOT / name, arcname=name)
print(f'FINAL EVIDENCE ARCHIVE: {archive_path}')
print(f'SHA256: {sha256_file(archive_path)}')
print('PENDING_USER_KAGGLE_RUN completed only when this cell prints PASS in a T4 session.')
